Load All Datasets

In [16]:
import pandas as pd

df_clean = pd.read_csv('/content/upi_fraud_dataset_clean.csv')
df_minmax = pd.read_csv('/content/upi_fraud_dataset_minmax.csv')
df_std = pd.read_csv('/content/upi_fraud_dataset_standardized.csv')
df_feat = pd.read_csv('/content/upi_fraud_feature_engineered.csv')

Check Shape

In [15]:
print(df_clean.shape)
print(df_minmax.shape)
print(df_std.shape)
print(df_feat.shape)

(11744, 22)
(11744, 22)
(11744, 22)
(11744, 22)


Remove Duplicate Columns

In [5]:
df_combined = df_clean.copy()

# Add only NEW columns from feature dataset
for col in df_feat.columns:
    if col not in df_combined.columns:
        df_combined[col] = df_feat[col]

Add Scaled Features

In [6]:
for col in df_minmax.columns:
    if col not in df_combined.columns:
        df_combined[col + "_scaled"] = df_minmax[col]

Final Dataset

In [8]:
print(df_combined.shape)
df_combined.head()

(11744, 22)


,TransactionID,UserID,Amount,MerchantCategory,TransactionType,DeviceID,IPAddress,Latitude,Longitude,AvgTransactionAmount,...,UnusualAmount,NewDevice,FailedAttempts,PhoneNumber,BankName,Hour,Day,Month,Weekday,FraudFlag
0,842835309389,6867,0.858945,1,0,2528,8789,0.629791,0.387426,0.066893,...,1,0,0.50,2798604680,1,0.130435,0.700000,0.000000,0.000000,1
1,592863054785,527,0.909114,1,0,8933,2995,0.150171,0.215256,0.726628,...,1,0,0.50,3614149152,3,0.565217,0.000000,1.000000,0.000000,0
2,373481869464,9678,0.881492,1,1,5327,8074,0.427147,0.447314,0.875160,...,0,1,0.75,912661191911,3,0.217391,0.033333,0.500000,0.166667,0
3,285572156436,2760,0.397027,4,1,6393,5726,0.367211,0.727168,0.834874,...,1,0,1.00,916358088125,5,0.695652,0.433333,0.833333,0.666667,0
4,874207772966,4823,0.437589,3,0,5403,2641,0.563272,0.535231,0.171863,...,1,0,0.00,5132838721,3,0.130435,0.500000,1.000000,0.166667,0


Prepare Data

In [9]:
y = df_combined['FraudFlag']

X = df_combined.drop(['FraudFlag', 'TransactionID', 'UserID', 'PhoneNumber'], axis=1, errors='ignore')

Add Train-Test Split Cell

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train

In [11]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight='balanced'
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 2172, number of negative: 7223
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009209 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2829
[LightGBM] [Info] Number of data points in the train set: 9395, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=300)

In [12]:
print(X_train.shape)
print(y_train.shape)

(9395, 18)
(9395,)


In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9101745423584504
[[1794   17]
 [ 194  344]]
